# ДЗ-16 (часть 2): интеграция модели с инструментами (LangChain tools + ReAct-агент)

Подключаем модель и даём ей доступ к внешним **инструментам** из [`tools.py`](tools.py):
`web_search`, `fact_check`, `calculator`.

## Провайдер LLM
Локально нет NVIDIA GPU, поэтому Qwen2.5-1.5B на CPU и медленная, и плохо держит формат ReAct.
Провайдер переключается флагом `LLM_PROVIDER` в `.env`:
- `openrouter` — облачные модели через OpenAI-совместимый API; по умолчанию бесплатная
  `qwen/qwen3-next-80b-a3b-instruct:free` (0$ за токены);
- `lmstudio` — локальный сервер [LM Studio](https://lmstudio.ai), без ключа и без интернета;
  по умолчанию `qwen/qwen3.5-9b` (в LM Studio также доступен `deepseek/deepseek-r1-0528-qwen3-8b` —
  это reasoning-модель, она заворачивает рассуждения в `<think>...</think>`, из-за чего парсинг
  `Action`/`Action Input` менее надёжен, чем у qwen3.5-9b);
- `transformers` — исходный вариант: своя загрузка HF-модели + опционально **дообученный**
  LoRA-адаптер из части 1 (`finetune_lora.ipynb`).

Крупные модели (в облаке или в LM Studio) гораздо надёжнее следуют формату
`Thought/Action/Action Input`, чем локальная Qwen2.5-1.5B на CPU — поэтому по умолчанию
используется `lmstudio`.

## Как модель использует инструменты (ReAct)
Маленькая модель сама по себе не вызывает функции. Мы оборачиваем её в цикл **ReAct**
(*Reasoning + Acting*): модель пишет рассуждение и выбирает действие в текстовом формате —

```
Thought: что нужно сделать
Action: имя_инструмента
Action Input: аргумент
Observation: <результат инструмента — подставляем мы>
... (повтор)
Final Answer: ответ пользователю
```

Мы парсим `Action`/`Action Input`, реально вызываем LangChain-tool через `.invoke(...)`,
возвращаем `Observation` обратно модели новым сообщением и повторяем, пока не появится
`Final Answer`.

> Инструменты определены через декоратор LangChain `@tool` (см. `tools.py`) — это и есть
> «LangChain Tools» из задания. Цикл ReAct реализован вручную поверх chat-messages: и для
> локальной модели, и для OpenAI-совместимого API (OpenRouter/LM Studio) это один и тот же код.
> Эквивалент на LangChain — в последней ячейке (закомментирован).

## Шаг 1. Установка

Базово нужны только `langchain`/инструменты + `openai` (клиент для OpenRouter и LM Studio).
`torch`/`transformers`/`peft` нужны, только если выберешь `LLM_PROVIDER=transformers`.

In [ ]:
# !pip install -q -U langchain langchain-community ddgs numexpr requests openai python-dotenv
# Нужны только при LLM_PROVIDER=transformers (локальная HF-модель):
# !pip install -q -U torch transformers peft accelerate

In [ ]:
import os, re
from dotenv import load_dotenv
from tools import TOOLS            # наши @tool из tools.py

load_dotenv()

LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "lmstudio")  # openrouter | lmstudio | transformers
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"     # только для LLM_PROVIDER == "transformers"
ADAPTER_DIR  = "qwen2.5-1.5b-lora-adapter"      # из части 1, только для "transformers"
LOAD_ADAPTER = False                            # True — если обучил адаптер в finetune_lora.ipynb

print("Провайдер LLM:", LLM_PROVIDER, "| инструменты:", [t.name for t in TOOLS])

## Шаг 2. Подключение модели

`openrouter`/`lmstudio` — OpenAI-совместимый чат-клиент (без загрузки весов).
`transformers` — своя загрузка HF-модели, опционально + LoRA-адаптер из части 1.

In [ ]:
if LLM_PROVIDER in ("openrouter", "lmstudio"):
    from openai import OpenAI

    if LLM_PROVIDER == "openrouter":
        # OpenRouter: OpenAI-совместимый API, один ключ на множество моделей.
        # Список бесплатных моделей: https://openrouter.ai/models?max_price=0
        client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
        MODEL = os.environ.get("OPENROUTER_MODEL", "qwen/qwen3-next-80b-a3b-instruct:free")
    else:
        # LM Studio: вкладка Developer -> Start Server (обычно http://localhost:1234/v1), ключ не нужен.
        client = OpenAI(
            base_url=os.environ.get("LMSTUDIO_BASE_URL", "http://localhost:1234/v1"),
            api_key=os.environ.get("LMSTUDIO_API_KEY", "lm-studio"),
        )
        # Имя модели должно совпадать с загруженной в LM Studio (см. "My Models" или GET /v1/models).
        # Локально доступны, например: qwen/qwen3.5-9b, deepseek/deepseek-r1-0528-qwen3-8b
        MODEL = os.environ.get("LMSTUDIO_MODEL", "qwen/qwen3.5-9b")

    print(f"Клиент готов: {LLM_PROVIDER} | модель: {MODEL} | base_url: {client.base_url}")
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    if device == "cpu":
        model = model.to("cpu")

    if LOAD_ADAPTER:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, ADAPTER_DIR)
        print("LoRA-адаптер подключён:", ADAPTER_DIR)
    else:
        print("Используется базовая модель (адаптер не подключён).")
    model.eval()
    print("Устройство:", device)

## Шаг 3. ReAct-агент поверх инструментов

In [ ]:
# Каждый инструмент принимает ОДИН строковый аргумент. Маппинг имени tool -> имя его аргумента,
# чтобы корректно вызвать tool.invoke({arg: action_input}).
PRIMARY_ARG = {"web_search": "query", "fact_check": "topic", "calculator": "expression"}
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

SYSTEM_PROMPT = """Ты — полезный ассистент с доступом к инструментам. Отвечай на русском.
Чтобы ответить, действуй строго в формате (по одному Action за раз):

Thought: твоё рассуждение
Action: имя инструмента (одно из: {names})
Action Input: только сам аргумент, без имени параметра и без скобок
  (Верно: Москва Неверно: fact_check(topic=Москва))
Observation: (результат подставит система)
... повторяй Thought/Action/Action Input/Observation столько раз, сколько нужно ...
Thought: теперь я знаю ответ
Final Answer: финальный ответ пользователю

Доступные инструменты:
{descriptions}

Не выдумывай Observation — его всегда подставляет система. Когда ответ готов, пиши Final Answer.""".format(
    names=", ".join(TOOLS_BY_NAME),
    descriptions="\n".join(f"- {t.name}: {t.description}" for t in TOOLS),
)

def _generate(messages, stop=("Observation:",), max_new_tokens=300) -> str:
    """Один шаг генерации ассистента поверх текущей истории messages."""
    if LLM_PROVIDER == "transformers":
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            stop_strings=list(stop), tokenizer=tokenizer,
        )
        text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    else:
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, stop=list(stop),
            max_tokens=max_new_tokens, temperature=0,
        )
        text = resp.choices[0].message.content or ""
    # У transformers стоп-строка остаётся в конце сгенерированного текста, у OpenAI-совместимых
    # API — нет (её обрезает сервер). Срезаем в обоих случаях, чтобы не задваивать "Observation:".
    return text.split("Observation:")[0].rstrip()

def run_agent(question: str, max_steps: int = 5, verbose: bool = True) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question}]

    for step in range(max_steps):
        chunk = _generate(messages)
        messages.append({"role": "assistant", "content": chunk})

        if "Final Answer:" in chunk:
            answer = chunk.split("Final Answer:")[-1].strip()
            if verbose: print(f"[шаг {step+1}] FINAL\n{chunk.strip()}\n")
            return answer

        m_act = re.search(r"Action:\s*(.+)", chunk)
        m_inp = re.search(r"Action Input:\s*(.+)", chunk)
        if not (m_act and m_inp):
            # модель не дала валидного действия — возвращаем что есть
            return chunk.strip()

        tool_name = m_act.group(1).strip().strip("`\"'")
        tool_arg  = m_inp.group(1).strip().strip("`\"'")
        if verbose: print(f"[шаг {step+1}] Action: {tool_name}({tool_arg})")

        tool = TOOLS_BY_NAME.get(tool_name)
        if tool is None:
            observation = f"Нет инструмента '{tool_name}'. Доступны: {list(TOOLS_BY_NAME)}"
        else:
            observation = str(tool.invoke({PRIMARY_ARG[tool_name]: tool_arg}))[:800]
        if verbose: print(f"        Observation: {observation[:160]}...\n")

        messages.append({"role": "user", "content": f"Observation: {observation}"})
    return "Достигнут лимит шагов без финального ответа."

## Шаг 4. Демонстрация в реальных сценариях

### Сценарий 1 — математика (calculator)

In [ ]:
print(run_agent("Посчитай, сколько будет 144 * 12 + 2 в степени 10."))

### Сценарий 2 — проверка факта (fact_check / Wikipedia)

In [ ]:
print(run_agent("Кто написал роман «Война и мир» и в каком веке жил автор?"))

### Сценарий 3 — свежая информация (web_search)

In [ ]:
print(run_agent("Найди, что нового в векторной БД Milvus версии 2.5."))

### Сценарий 4 — многоступенчатое рассуждение (fact_check → calculator)
Сначала найти факт, потом посчитать на его основе — два инструмента в одной цепочке.

In [ ]:
print(run_agent("Узнай высоту горы Эверест в метрах и переведи её в футы (1 метр = 3.281 фута)."))

## Альтернатива: высокоуровневый агент LangChain
Для моделей с нативным tool-calling (или через `ChatHuggingFace`) можно не писать цикл вручную,
а собрать агента стандартными средствами LangChain. Оставляю как референс.

In [ ]:
# from langchain.agents import AgentExecutor, create_react_agent
# from langchain_core.prompts import PromptTemplate
# from langchain_huggingface import HuggingFacePipeline
# from transformers import pipeline
#
# pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,
#                 max_new_tokens=256, return_full_text=False)
# llm = HuggingFacePipeline(pipeline=pipe)
# prompt = PromptTemplate.from_template(
#     "Answer the question using tools.\n{tools}\nTool names: {tool_names}\n"
#     "Use format Thought/Action/Action Input/Observation/Final Answer.\n"
#     "Question: {input}\n{agent_scratchpad}")
# agent = create_react_agent(llm, TOOLS, prompt)
# executor = AgentExecutor(agent=agent, tools=TOOLS, verbose=True, handle_parsing_errors=True)
# print(executor.invoke({"input": "Сколько будет 144*12 + 2^10?"}))

## Выводы
- Инструменты оформлены через LangChain `@tool` (`tools.py`) — у каждого есть имя, описание и схема,
  по которым агент решает, что вызвать.
- Цикл **ReAct** связывает модель с инструментами через chat-messages: модель рассуждает → выбирает
  действие → получает результат новым сообщением → продолжает, пока не сформирует ответ.
- Так LLM получает то, чего нет в весах: **актуальные данные** (web_search), **достоверные факты**
  (Wikipedia) и **точные вычисления** (calculator).
- Провайдер (`LLM_PROVIDER`) не меняет логику агента — меняется только то, откуда берётся ответ
  ассистента. На практике облачные/LM Studio модели (9B+) держат формат `Thought/Action/Action Input`
  заметно надёжнее, чем локальная Qwen2.5-1.5B на CPU; для 1.5B срывы формата — ожидаемая демонстрация
  границ маленькой модели, а не баг цикла.